# pii

> whether a document is somebody's business, decided by arithmetic rather than by a model

In [ ]:
#| default_exp pii

PII detection uses checksummed patterns: 0.962 precision at recall 1.000 (`evals/pii.py`). API keys (`secret`) gate too. Modes live on `ask(pii=…)`: see [ask](02_ask.ipynb).


In [ ]:
#| export
import re
from fastcore.all import AttrDict, L

## What counts

- `MAX_SCAN`: sample both ends of a long document (headers hold account numbers).
- `DENSE`: matches per thousand chars; for callers separating a signature block from a customer list, not `has_pii`.
- `NER_CHARS` (20,000, from `extract`): cap on the opt-in name pass (`ner=True`).


In [ ]:
#| export
MAX_SCAN, DENSE = 200_000, 1.0

In [ ]:
#| export
def luhn(s:str) -> bool:
    "The check digit every payment card carries. Sixteen digits that fail it are not a card."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) < 12: return False
    tot, parity = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == parity: d *= 2; d -= 9 if d > 9 else 0
        tot += d
    return tot % 10 == 0

def _iban_ok(s:str) -> bool:
    "IBAN's mod-97 check: move the country prefix to the end, letters to digits, remainder must be 1."
    s = re.sub(r'[^A-Za-z0-9]', '', s).upper()
    if not (15 <= len(s) <= 34): return False
    t = s[4:] + s[:4]
    try: n = int(''.join(str(int(c, 36)) for c in t))
    except ValueError: return False
    return n % 97 == 1

def _nhs_ok(s:str) -> bool:
    "The UK NHS number's mod-11 check digit. Ten digits in a row are otherwise just ten digits."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 10: return False
    tot = sum(d * (10 - i) for i, d in enumerate(ds[:9]))
    chk = 11 - tot % 11
    return chk != 10 and (0 if chk == 11 else chk) == ds[9]

def _ssn_ok(s:str) -> bool:
    "A US SSN's structurally impossible cases, which is as much as arithmetic can say about one."
    ds = re.sub(r'\D', '', s)
    if len(ds) != 9: return False
    a, b, c = ds[:3], ds[3:5], ds[5:]
    return a not in ('000', '666') and a[0] != '9' and b != '00' and c != '0000'

Checksums lift precision from 0.664 to 0.948 at recall 1.000 on 400 documents half carrying lookalikes (101/200 → 11/200 false positives, `evals/pii.py`).


In [ ]:
#| export
#: kind -> (pattern, validator or None). Spans de-overlapped longest-first.
PATTERNS = {
    'email':   (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', None),
    'card':    (r'\b(?:\d[ -]?){13,19}\b', luhn),
    'iban':    (r'\b[A-Z]{2}\d{2}[ ]?(?:[A-Z0-9]{4}[ ]?){2,7}[A-Z0-9]{1,4}\b', _iban_ok),
    'ssn':     (r'\b\d{3}-\d{2}-\d{4}\b', _ssn_ok),
    'nhs':     (r'\b\d{3}[ -]?\d{3}[ -]?\d{4}\b', _nhs_ok),
    'phone':   (r'\+\d{1,3}[ .-]?\(?\d{1,5}\)?[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\(\d{2,5}\)[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b0\d{1,4}[ .-]\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b\d{3}-\d{3}-\d{4}\b'
                r'|\b(?:phone|tel|telephone|mobile|cell|fax)\b\W{0,8}\+?[\d ().-]{7,20}\d', None),
    'ip':      (r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', None),
    'dob':     (r'\b(?:date of birth|dob|born)\b\W{0,12}(?:\d{1,4}[/-]\d{1,2}[/-]\d{1,4}|\d{1,2} \w+ \d{4})', None),
    'passport':(r'\b(?:passport(?:\s*(?:no|number|#))?)\W{0,6}[A-Z0-9]{6,9}\b', None),
    'licence': (r'\b(?:driver.?s? licen[cs]e|dl)(?:\s*(?:no|number|#))?\W{0,6}[A-Z0-9]{5,20}\b', None),
    'account': (r'\b(?:account|acct|a/c)(?:\s*(?:no|number|#))?\W{0,6}\d{6,17}\b', None),
    'sortcode':(r'\b(?:sort\s*code)\W{0,6}\d{2}[- ]?\d{2}[- ]?\d{2}\b', None),
    'secret':  (r'\b(?:sk-[A-Za-z0-9_-]{16,}|ghp_[A-Za-z0-9]{20,}|xox[baprs]-[A-Za-z0-9-]{10,}|AKIA[0-9A-Z]{16}|AIza[0-9A-Za-z_-]{35})\b', None),
    'medical': (r'\b(?:patient (?:id|number|name)|nhs number|medical record(?:\s*(?:no|number|#))?|mrn\W{0,6}\w+)\b', None),
    # require a street name between number and suffix (0.948 vs 0.746 without, evals/pii.py)
    'address': (r'\b\d{1,5}[A-Za-z]?[ ,]+(?:[A-Z][A-Za-z.\'-]+[ ,]+){1,3}'
                r'(?:Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Drive|Boulevard|Blvd|Close|Court|Ct'
                r'|Crescent|Way|Place|Terrace|Square|Sq|Gardens|Grove|Row|Walk)\b\.?'
                r'|\b[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}\b'
                r'|\b[A-Z]{2} \d{5}(?:-\d{4})?\b', None),
}
#: `person` is the one kind arithmetic cannot find, and the only one gated behind `ner=True`.
IDENTIFYING = frozenset({'email', 'card', 'iban', 'ssn', 'nhs', 'phone', 'dob', 'passport', 'licence', 'account',
                         'sortcode', 'medical', 'secret', 'address', 'person'})
CASED = frozenset({'address'})
_COMPILED = {k: (re.compile(p, 0 if k in CASED else re.I), v) for k, (p, v) in PATTERNS.items()}


## Where it is

In [ ]:
#| export
def _scan_parts(text:str, mx:int=MAX_SCAN) -> list:
    """Both ends of a long document as `(offset, part)` (headers/footers hold identifiers)."""
    text = str(text or '')
    if len(text) <= mx: return [(0, text)]
    half = mx // 2
    return [(0, text[:half]), (half + 1, text[-half:])]

def _scan_text(text:str, mx:int=MAX_SCAN) -> str:
    "The part of a long document worth scanning, joined for measurement. Offsets follow `_scan_parts`."
    return '\n'.join(p for _, p in _scan_parts(text, mx))

def person_spans(text:str,     # what to scan
                 mx:int=None,  # chars handed to the extractor; None -> `extract.NER_CHARS`
) -> L:
    """Honorific-anchored personal names. Off by default; costs an entity pass."""
    from vishalakshi.extract import NER_CHARS, _noun_ents
    text = str(text or '')[:mx or NER_CHARS]
    out = []
    for surface, label in _noun_ents(text):
        if label != 'PERSON': continue
        # `_noun_ents` collapses whitespace, so match across the line breaks a PDF leaves inside
        # a name -- and on word boundaries, or `Ross` masks the middle of `Rossini`.
        rx = r'\b' + r'\s+'.join(map(re.escape, surface.split())) + r'\b'
        out += [(m.start(), m.end(), 'person', m.group(0)) for m in re.finditer(rx, text)]
    return L(out)

def pii_spans(text:str,          # what to scan
              kinds=None,        # restrict to these kinds; None -> every pattern
              mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
              ner:bool=False,    # also look for names, which no pattern can find
) -> L:
    """Every match, as `(start, end, kind, text)`, longest first and never overlapping."""
    want = set(kinds or (*_COMPILED, 'person'))
    found = []
    for off, part in _scan_parts(text, mx):
        for kind, (rx, ok) in _COMPILED.items():
            if kind not in want: continue
            for m in rx.finditer(part):
                if ok is not None and not ok(m.group(0)): continue
                found.append((off + m.start(), off + m.end(), kind, m.group(0)))
        if ner and 'person' in want:
            found += [(off + s, off + e, k, v) for s, e, k, v in person_spans(part)]
    found.sort(key=lambda s: (s[0] - s[1], s[0]))       # longest first, then leftmost
    out, taken = [], []
    for s, e, kind, val in found:
        if any(s < te and ts < e for ts, te in taken): continue
        taken.append((s, e))
        out.append((s, e, kind, val))
    return L(sorted(out))


In [ ]:
#| export
def pii_report(text:str,          # what to scan
               kinds=None,        # restrict to these kinds; None -> every pattern
               mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
               ner:bool=False,    # also look for names
) -> AttrDict:
    """Spans found and whether they tip `has_pii` (IDENTIFYING kinds only)."""
    spans, counts = pii_spans(text, kinds, mx, ner=ner), {}
    for _, _, k, _ in spans: counts[k] = counts.get(k, 0) + 1
    n = len(_scan_text(text, mx))
    ident = {k: v for k, v in counts.items() if k in IDENTIFYING}
    # `n` and `density` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
    n_arith = len(spans) - counts.get('person', 0)
    return AttrDict(has_pii=bool(ident), kinds=counts, identifying=ident, n=n_arith,
                    n_person=counts.get('person', 0), scanned=n, scanned_ner=bool(ner),
                    density=round(1000 * n_arith / max(n, 1), 3), spans=spans)


`has_pii` counts only `IDENTIFYING`. IP addresses remain reportable but do not trigger the gate.


In [ ]:
#| export
def redact(text:str,       # the text to mask
           spans=None,     # spans from `pii_spans`; recomputed over the whole of `text` when None
           kinds=None,     # restrict to these kinds
           mask:str=None,  # what to put in place of a match; None -> `[KIND]`
           ner:bool=False, # also mask names
) -> str:
    """Mask matched spans. Names only with `ner=True`."""
    out = str(text or '')
    if spans is None: spans = pii_spans(out, kinds, mx=len(out), ner=ner)
    for s, e, kind, _ in sorted(spans, reverse=True):
        out = out[:s] + (mask if mask is not None else f'[{kind.upper()}]') + out[e:]
    return out


## Asking the vault

`Vault.pii` is document-level. `pii_ctx` gates an answer over the sections retrieval chose.


In [ ]:
#| export
from vishalakshi.core import Vault
from fastcore.all import patch

def redact_obj(o, kinds=None, ner:bool=False):
    "`redact` over the strings inside a nested dict or list: what a structured answer is."
    if isinstance(o, str):  return redact(o, kinds=kinds, ner=ner)
    if isinstance(o, dict): return {k: redact_obj(v, kinds, ner) for k, v in o.items()}
    if isinstance(o, list): return [redact_obj(v, kinds, ner) for v in o]
    return o

@patch
def pii(self:Vault,
        ref,                 # a doc_id, source, title or path: whatever `document` takes
        max_chars:int=MAX_SCAN,
        ner:bool=False,      # also look for names, except on code, where identifiers are not names
) -> AttrDict:
    "Whether one whole document is somebody's business, and what in it says so."
    d = self.document(ref, max_chars=max_chars)
    # `scanned_ner` then reports False, which is the honest answer: nothing looked for a name here
    r = pii_report(d.text, ner=ner and (d.get('kind') or '') != 'code')
    override = (self.marks(d.get('doc_id')) or {}).get('pii_override') if d.get('doc_id') else None
    r.detected, r.override = r.has_pii, override
    if override == 'clear': r.has_pii = False
    elif override == 'force': r.has_pii = True
    r.doc_id, r.title, r.source = d.get('doc_id'), d.get('title'), d.get('source')
    return r

@patch
def mark_not_pii(self:Vault, ref, clear:bool=True, reason:str='') -> dict:
    "Clear a false-positive PII decision (`pii_override='clear'`), or restore automatic detection."
    return self.mark(ref, pii_override='clear' if clear else None,
                     pii_reason=(reason or None) if clear else None)

@patch
def mark_pii(self:Vault, ref, force:bool=True, reason:str='') -> dict:
    "Force a document private even when arithmetic finds nothing (names, addresses, whole PDFs)."
    return self.mark(ref, pii_override='force' if force else None,
                     pii_reason=(reason or None) if force else None)

def pii_ctx(ctx, ner:bool=False) -> AttrDict:
    "The report for an assembled context, which is what a policy has to gate on."
    parts = [str(getattr(r, 'text', None) or (r.get('text') if isinstance(r, dict) else '') or '')
             for r in (list(ctx.get('results') or []) + list(ctx.get('related') or []))]
    return pii_report('\n\n'.join(parts), ner=ner)

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Using it

In [ ]:
report = pii_report("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00, account number 12345678.
Server 10.0.0.14 returned 500. Order number 4471000012345678.
""")
report.has_pii, report.identifying, report.n

(True, {'email': 1, 'phone': 1, 'card': 1, 'sortcode': 1, 'account': 1}, 6)

In [ ]:
from fastcore.test import test_eq
test_eq(report.has_pii, True)
test_eq('card' in report.identifying, True)      # passes Luhn
test_eq(report.kinds.get('ip'), 1)               # reported...
test_eq('ip' in report.identifying, False)       # ...but a log is not somebody's private life

In [ ]:
print(redact("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00.
""").strip())

Invoice 4471 for Ada Lovelace <[EMAIL]>, [PHONE].
Card [CARD], [SORTCODE].


Names are opt-in (`ner=True`) and honorific-anchored: `Dr Charles Babbage` matches, bare `Ada Lovelace` does not. Read `scanned_ner` before a zero. Street lines are arithmetic like the rest.


In [ ]:
#| hide
from fastcore.test import test_eq

# secrets gate: a key alone is identifying
test_eq(pii_report('export OPENAI_API_KEY=sk-abcdefghijklmnopqrstuvwxyz123456').has_pii, True)
test_eq('secret' in pii_report('token ghp_abcdefghijklmnopqrst').identifying, True)

# medical: research prose is not a medical record; a patient id is
test_eq(pii_report('This paper diagnoses a failure mode in the prescription of learning rates').has_pii, False)
test_eq(pii_report('Patient id 44291 was discharged yesterday.').has_pii, True)

# nested redact for structured answers
test_eq(redact_obj({'email': 'a@b.co', 'n': 1}), {'email': '[EMAIL]', 'n': 1})
test_eq(redact_obj(['a@b.co', 3])[0], '[EMAIL]')


# a street line is what makes "John Smith, 12 Elm Street" private: no pattern finds the name
test_eq(pii_report('John Smith, 12 Elm Street').identifying, {'address': 1})
test_eq(pii_report('900 Market St, San Francisco CA 94103').has_pii, True)
test_eq(pii_report('The registered office is 221B Baker Street, London NW1 6XE').kinds['address'], 2)
# a numbered heading is not an address, which is the whole precision argument
for _t in ('Chapter 4 Court decisions', 'Table 3 Road traffic figures', 'Figure 2 Way of working'):
    test_eq(pii_report(_t).has_pii, False)

In [ ]:
#| hide
# 1. off by default, capped at NER_CHARS when on
from vishalakshi.extract import NER_CHARS
_sig = 'Dr Charles Babbage signed it.'
test_eq(pii_report(_sig).has_pii, False)                       # names are not looked for
test_eq(pii_report(_sig, ner=True).identifying, {'person': 1})   # ...until asked
test_eq(pii_report('Ada Lovelace signed it.', ner=True).has_pii, False)   # an honorific is the anchor
test_eq(pii_report('x'*NER_CHARS + ' ' + _sig, ner=True).kinds.get('person'), None)   # past the cap

# 2. `scanned_ner` keeps "none found" apart from "not looked for"
test_eq(pii_report('nothing here').scanned_ner, False)
test_eq(pii_report('nothing here', ner=True).scanned_ner, True)

# names are masked once asked for, and not before
test_eq('[PERSON]' in redact(_sig), False)
test_eq(redact(_sig, ner=True), 'Dr [PERSON] signed it.')

# the seam: `_scan_parts` keeps the halves apart, so an honorific at the end of one and a
# capitalised pair at the start of the next is not a person who was never in the document
_half = 4000
_doc = 'x'*(_half-3) + ' Dr' + 'm'*5000 + 'Charles Babbage wrote it.' + 'z'*(_half-25)
test_eq(pii_report(_doc, mx=8000, ner=True).kinds.get('person'), None)

# `density` and `n` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
_names = 'Dr Ada Lovelace met Dr Charles Babbage. '*5
test_eq((pii_report(_names).density, pii_report(_names, ner=True).density), (0.0, 0.0))
test_eq(pii_report(_names, ner=True).n_person, 10)

In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'p.db', offline=True)
v.add('A letter about Jane, and what she said on Tuesday.', title='letter', source='/inbox/letter.md')
r = v.pii('/inbox/letter.md')
test_eq(r.has_pii, False)
v.mark_pii('/inbox/letter.md', reason='address book')
test_eq(v.pii('/inbox/letter.md').has_pii, True)
test_eq(v.pii('/inbox/letter.md').override, 'force')
v.mark_not_pii('/inbox/letter.md', reason='my own draft')
test_eq(v.pii('/inbox/letter.md').has_pii, False)
test_eq(v.pii('/inbox/letter.md').override, 'clear')

In [ ]:
#| hide
# 3. no NER on code: an identifier is not a name, and a report must not claim it looked
v.add('# Dr Charles Babbage wrote this\ndef f(): pass', title='mod', source='/m.py', kind='code')
_c = v.pii('/m.py', ner=True)
test_eq((_c.scanned_ner, _c.has_pii), (False, False))

v.add(_sig, title='signed', source='/inbox/signed.md')
_p = v.pii('/inbox/signed.md', ner=True)
test_eq((_p.scanned_ner, _p.has_pii, _p.identifying), (True, True, {'person': 1}))
test_eq(v.pii('/inbox/signed.md').scanned_ner, False)   # the default is still arithmetic only

A number that fails its checksum is not the thing the checksum protects.

In [ ]:
test_eq(pii_report('Order 4111 1111 1111 1112 shipped').has_pii, False)   # fails Luhn
test_eq(pii_report('Card 4111 1111 1111 1111 charged').has_pii, True)     # passes it
test_eq(pii_report('The build takes 20 minutes and costs nothing.').has_pii, False)

In [ ]:
#| hide
long_doc = 'Account number 12345678\n' + ('filler text. ' * 40_000) + '\nsigned, ada@example.com'
r = pii_report(long_doc)
test_eq(r.has_pii, True)
test_eq(sorted(r.identifying), ['account', 'email'])
test_eq(r.scanned <= MAX_SCAN + 1, True)

# ...but a *report* may sample and a redaction may not
masked = redact(long_doc)
assert 'ada@example.com' not in masked, masked[-80:]
assert '12345678' not in masked, masked[:80]
test_eq(masked.count('filler text. '), 40_000)      # and nothing in between was moved
test_eq(pii_report(masked).has_pii, False)

In [ ]:
#| hide
one = pii_report('4111 1111 1111 1111')
test_eq(one.n, 1)
test_eq(list(one.kinds), ['card'])

Phones need a separator or trunk prefix: bare digit runs are not phones (0.948 → 0.962 precision, `evals/pii.py`).


In [ ]:
#| hide
# What must and must not put a document on the local-only path
cases = [
    ('Order 4111 1111 1111 1112 shipped',                 False),   # fails Luhn
    ('Card 4111 1111 1111 1111 charged',                  True),
    ('phone 020 7946 0958',                               True),
    ('+44 20 7946 0958',                                  True),
    ('call (555) 123-4567',                               True),
    ('555-123-4567',                                      True),
    ('ada@example.com',                                   True),
    ('The build takes 20 minutes and costs nothing.',     False),
    ('Run 2024 1000 2000 3000 through the pipeline',      False),
    ('Release 1.2.3 shipped on 2024-05-01 with 400 tests', False),
    ('Server 10.0.0.14 returned 500',                     False),   # reported, not identifying
    ('commit 8f3a2b1 touched 120 lines in 14 files',      False),
]
for text, want in cases: test_eq((text, pii_report(text).has_pii), (text, want))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()